# Assignment 3

It is reccomended to create a virtual environment before executing the code in this notebook.

After creating a new virtual environment, run the code below to install the dependencies

In [ ]:
!pip install -r requirements.txt

__IMPORTANT__: Ollama needs to be installed in order for this code to run. Additionally, Ollama has to be serving so that the this code can access the model. In order to install Ollama on your device, visit Ollama's [download page](https://ollama.com/download) and select the install method for your operating system.

Since we are using Ollama's cloud models for remote computing due to hardware limitations, you need to create an Ollama account [here](https://ollama.com/signup) (free).
Rate limits apply for the free tier, but are sufficient for running this code.

Alternetively, you can change the model in the variable below to any local ollama model, and you can skip the signup and signin steps. However, depdending on your hardware, running the code may take a very long time.

In [91]:
model = 'ministral-3:14b-cloud' # change this to a non-cloud model if you want to run a local model and skip signing up and signing in

In [ ]:
!ollama serve # start running ollama
!ollama pull ministral-3:14b-cloud # this pulls ministral 3 14b metadata for the ollama cloud model

The last step in setting up is signing in to Ollama to get access to the cloud models. You can click the link you see in the cell output to login. After logging in, you can access the cloud models.

In [ ]:
!ollama signin

## Predicting song lyrics genres using LLMs

First, we import the required Python modules and load in the .csv files as dataframes

In [100]:
import ollama
from pathlib import Path
import pandas as pd
from sklearn.metrics import (precision_score, recall_score, f1_score, classification_report, confusion_matrix)
import time

test = pd.read_csv(Path.cwd() / "data" / "genreLyrics_test.csv", sep="\t") # the csv file uses tabs as seperators --> sep="\t"
train = pd.read_csv(Path.cwd() / "data" / "genreLyrics_train.csv", sep="\t") # the csv file uses tabs as seperators --> sep="\t"

print(test.shape) # the testing set contains 1500 rows
print(train.shape) # the training set contains 5000 rows

(1500, 3)
(5000, 3)


### Zero-shot prompting

For zero shot prompting, we ignore the training set and see how well the model performs at assigning a specific genre to the lyrics of a song in the test set.

The code below prompts ministral 3 to classify the genre considering the lyrics. It adds the results to a new column `predicted_results` in the dataframe `zeroshot`

#### ⚠️ This will take a long time!
Alternetively, the results have been saved to `/results` in this repo, and are imported later in the code.

In [4]:
zeroshot = test # make a copy of the test set for zero-shot promtping

genres = zeroshot['genre'].unique() # create a list of genres found in the test set to feed into the prompt

def assign_genre_zeroshot(lyrics):
    
    prompt = f"""
    You are a world‑renowned musicologist who has curated millions of songs and knows the subtle differences between musical genres.
    You are presented with song lyrics.
    First, analyze the lyrical themes and instrumentation clues.
    You have to identify what music genre the song lyrics correspond to.
    Make sure to only respond with one genre from this list: {genres}, WITH THE EXACT SAME SPELLING.
    LITERALLY RESPOND WITH 'Other' IF NONE OF THE GENRES MATCH, DO NOT MAKE UP ANY GENRES OUTSIDE OF THIS LIST.
    REFRAIN FROM USING ASTERIKS IN THE ANSWER, AND ALWAYS ONLY ANSWER WITH JUST THE GENRE FROM THE LIST.
    REMINDER: RESPONDS SOLELY WITH THE GENRE.
    Lyrics: 
    <<<{lyrics}>>>
    """
    
    response = ollama.chat(model=model, 
              messages=[{'role': 'user', 'content': prompt,}],
              )
    
    print(response['message']['content'])
    
    return response['message']['content']
    
zeroshot['predicted_genre'] = zeroshot['lyrics'].apply(assign_genre_zeroshot) # apply the function for every lyric in the test set!

Hip-Hop
Country
Indie
Other
Country
Pop
Indie
Indie
Folk
R&B
Country
Pop
Country
Indie
Folk
Metal
Folk
Pop
Jazz
Folk
Hip-Hop
Hip-Hop
Indie
Indie
Indie
Indie
Indie
Indie
Indie
Folk
Folk
Pop
Hip-Hop
Indie
Pop
Country
Indie
Indie
Pop
Indie
Metal
Indie
Pop
Indie
Country
Pop
Horrorcore
R&B
Pop
Indie
R&B
Indie
Indie
Indie
Indie
Indie
Indie
Hip-Hop
Folk
Country
Pop
Indie
Pop
Rock
Indie
Indie
Country
Indie
Country
Folk
Indie
Indie
Folk
Rock
Hip-Hop
Country
Indie
Pop
Hip-Hop
Indie
Indie
Folk
Indie
Folk
Indie
Pop
Indie
Pop
Pop
Pop
Hip-Hop
Indie
Pop
Folk
Hip-Hop
Indie
Pop
Folk
Indie
Hip-Hop
Indie
Metal
Indie
Indie
Gospel
Country
Indie
Folk
Hip-Hop
Indie
Folk
Gospel
Folk
Indie
Metal
Country
Hip-Hop
Pop
Metal
R&B
Folk
Folk
Indie
Pop
Pop
Indie
Indie
Indie
Indie
Hip-Hop
Hip-Hop
Indie
Indie
Indie
Pop
Folk
Indie
Gospel
Indie
Pop
Folk
Indie
Indie
Indie
Indie
Indie
Hip-Hop
Indie
Folk
Folk
Indie
Blues
Hip-Hop
Folk
Indie
Indie
Hip-Hop
Pop
Folk
Hip-Hop
Hip-Hop
Indie
Indie
Indie
Folk
Metal
Folk
Metal
Gospel


In [6]:
zeroshot.to_csv("results/zeroshot_ministral-14B.csv", index=False) # save the results locally for reuse without calling the LLM again

In [7]:
zeroshot = pd.read_csv("results/zeroshot_ministral-14B.csv") # load in the saved results of the zero-shot prompting
zeroshot.head(20)

,Unnamed: 0,genre,lyrics,predicted_genre
0,253513,Hip-Hop,"State Property Music\nuh, holla, uh...... yeah...",Hip-Hop
1,218012,Country,Well I really had a ball last night\nI held al...,Country
2,91614,Rock,[lyrics:Stefan Ruiters]\nsolar child is burnin...,Indie
3,320535,Electronic,La milonga.\nLadies and gentlemen\nA man.\nMil...,Other
4,248213,Pop,Anybody ever tell you that you're not whole\nA...,Country
5,130989,Indie,I give myself up every day\nTo fight this woe ...,Pop
6,281194,Rock,We were in a park and I could jump so far\nSee...,Indie
7,312266,Rock,Guilty As A Saint\n(Prophet/klipschutz)\nI was...,Indie
8,103107,Folk,Once I had a darling mother though I can't rec...,Folk
9,116502,Rock,"I am a vision, I am justice\nNever thought tha...",R&B


#### Evaluating the zero-shot prompting performance 

Below, we specify the true values and the predictions, and use them to make a classification report using sklearn. This gives us an overview of performence (precision, recall, f1) per genre as well as the total average weighted metrics.

In [108]:
true = zeroshot['genre'] # get the 'true' values
pred = zeroshot['predicted_genre'] # get the 'true' values

# create a classification report with sklearn
report = classification_report(
    true,
    pred,
    labels=genres,    
    target_names=genres,
    output_dict=True
)

report = pd.DataFrame(report).transpose  # dict to df, transpose, because it looks better :) 
report.to_csv("results/zeroshot_report.csv") # save it
report

,precision,recall,f1-score,support
Hip-Hop,0.631579,0.809816,0.709677,163.0
Country,0.310924,0.415730,0.355769,89.0
Rock,0.703704,0.029008,0.055718,655.0
Electronic,0.142857,0.017544,0.031250,57.0
Pop,0.351351,0.254902,0.295455,255.0
Indie,0.013473,0.562500,0.026316,16.0
Folk,0.046053,0.411765,0.082840,17.0
R&B,0.041667,0.043478,0.042553,23.0
Metal,0.691176,0.311258,0.429224,151.0
Jazz,0.125000,0.023256,0.039216,43.0


### Few-shot prompting

For few-shot prompting, we include one example lyrics per genre to gice context to the model.

In [85]:
# check what genres appear in the training set
train['genre'].unique()

array(['Rock', 'Country', 'Electronic', 'Pop', 'Hip-Hop', 'Metal',
       'Indie', 'R&B', 'Folk', 'Jazz', 'Other'], dtype=object)

In [98]:
def assign_genre_fewshot(lyrics, examples):
    """
    This function takes lyrics (string) and examples (dict with example lyrics as values and genre as key) for few-shot prompting
    Returns a genre classification made by ministral-3:14b (string)
    """
    
    prompt = f"""
    You are a world‑renowned musicologist who has curated millions of songs and knows the subtle differences between musical genres.
    You are presented with song lyrics.
    First, analyze the lyrical themes and instrumentation clues.
    You have to identify what music genre the song lyrics correspond to.
    Make sure to only respond with one genre from this list: {genres}, WITH THE EXACT SAME SPELLING.
    LITERALLY RESPOND WITH 'Other' IF NONE OF THE GENRES MATCH, DO NOT MAKE UP ANY GENRES OUTSIDE OF THIS LIST.
    REFRAIN FROM USING ASTERIKS IN THE ANSWER, AND ALWAYS ONLY ANSWER WITH JUST THE GENRE FROM THE LIST.
    REMINDER: RESPONDS SOLELY WITH THE GENRE.

    Below are some examples of lyrics with their corresponding genre:

    <<<Rock>>> : {examples['Rock']}

    <<<Country>>> : {examples['Country']}
    
    <<<Electronic>>> : {examples['Electronic']}
    
    <<<Pop>>> : {examples['Pop']}
    
    <<<Hip-Hop>>> : {examples['Hip-Hop']}
    
    <<<Metal>>> : {examples['Metal']}
       
    <<<Indie>>>: : {examples['Indie']}
    
    <<<R&B>>> : {examples['R&B']}
    
    <<<Folk>>> : {examples['Folk']}
    
    <<<Jazz>>> : {examples['Jazz']}
    


    
    Below is the lyrics you have to classify: 
    <<<{lyrics}>>>
    """

    # initialize variables for handling internal server errors on Ollama's servers
    max_retries = 5
    base_delay=2

    # the ollama code is embedded in a for loop and try block to catch internal server errors and try again with a delay
    for attempt in range(1, max_retries + 1):
        try:
            response = ollama.chat(model=model, 
                        messages=[{'role': 'user', 'content': prompt,}],
                        options={"num_predict": 10}
                      )
            
            print(response['message']['content'])
            
            return response['message']['content']

        except Exception as e:
            print(e)
            if attempt < max_retries:
                delay = base_delay * (2 ** (attempt - 1))  # 1s → 2s → 4s
                print(f"Retry {attempt}/{max_retries} for 500 error. Waiting {delay}s...")
                time.sleep(delay)
                continue
            else:
                print(f"\nMax retries ({max_retries}) exhausted. Last error: {str(e)[:100]}")
                return "Failed"
            

In [102]:
fewshot = test # copy the test set to new df

# create a dictionary with one example lyrics per genre (3rd song per genre, randomly chosen)
examples = {}
for genre in genres:
    examples[genre] = train[train['genre'] == genre].iloc[3,2]

# run the function on test set and add a column with the LLM predicted genre
fewshot['predicted_genre'] = fewshot['lyrics'].apply(assign_genre_fewshot, examples=examples)

Hip-Hop
Indie
Indie
Other
Pop
Pop
Indie
Indie
Folk
Rock
Pop
Pop
Indie
Indie
Pop
Metal
Folk
Pop
Indie
Indie
Hip-Hop
Hip-Hop
Indie
Pop
Indie
Indie
Indie
Metal
Indie
Folk
Folk
Pop
Hip-Hop
Indie
Pop
Pop
Pop
Indie
Pop
Indie
Metal
Indie
Pop
Metal
Pop
Gospel
Metal
R&B
Pop
Indie
R&B
Metal
Metal
Indie
Rock
Pop
Indie
Hip-Hop
Folk
Pop
Pop
Indie
Pop
Pop
Indie
Indie
Pop
Indie
Rock
Country
Rock
Pop
Indie
Metal
Hip-Hop
Pop
Indie
Pop
Hip-Hop
Other
Indie
Folk
Indie
Indie
Indie
Pop
Rock
Pop
Pop
Pop
Hip-Hop
Metal
Pop
Indie
Hip-Hop
Indie
Pop
Indie
Indie
Hip-Hop
Indie
Metal
Indie
Indie
Gospel
Country
Indie
Indie
Hip-Hop
Metal
Indie
Gospel
Indie
Indie
Metal
Country
Hip-Hop
Gospel
Metal
R&B
Indie
Indie
Pop
Pop
Pop
Rock
Rock
Indie
Indie
Hip-Hop
Hip-Hop
Indie
Rock
Indie
Pop
Rock
Indie
Folk
Indie
Folk
Folk
Rock
Pop
Indie
Indie
Metal
Hip-Hop
Indie
Pop
Indie
Indie
Blues
Hip-Hop
Folk
Metal
Indie
Hip-Hop
Pop
Country
Hip-Hop
Hip-Hop
Indie
Indie
Indie
Gospel
Metal
Folk
Metal
Folk
Indie
Indie
Pop
Indie
Pop
Indie
Hip-H

,Unnamed: 0,genre,lyrics,predicted_genre
0,253513,Hip-Hop,"State Property Music\nuh, holla, uh...... yeah...",Hip-Hop
1,218012,Country,Well I really had a ball last night\nI held al...,Indie
2,91614,Rock,[lyrics:Stefan Ruiters]\nsolar child is burnin...,Indie
3,320535,Electronic,La milonga.\nLadies and gentlemen\nA man.\nMil...,Other
4,248213,Pop,Anybody ever tell you that you're not whole\nA...,Pop
5,130989,Indie,I give myself up every day\nTo fight this woe ...,Pop
6,281194,Rock,We were in a park and I could jump so far\nSee...,Indie
7,312266,Rock,Guilty As A Saint\n(Prophet/klipschutz)\nI was...,Indie
8,103107,Folk,Once I had a darling mother though I can't rec...,Folk
9,116502,Rock,"I am a vision, I am justice\nNever thought tha...",Rock


In [103]:
fewshot.to_csv("results/fewshot_ministral-14B.csv", index=False) # save the results locally for reuse without calling the LLM again

In [105]:
fewshot = pd.read_csv("results/fewshot_ministral-14B.csv") # load the saved results, removing the need for inference
fewshot.head(20)

,Unnamed: 0,genre,lyrics,predicted_genre
0,253513,Hip-Hop,"State Property Music\nuh, holla, uh...... yeah...",Hip-Hop
1,218012,Country,Well I really had a ball last night\nI held al...,Indie
2,91614,Rock,[lyrics:Stefan Ruiters]\nsolar child is burnin...,Indie
3,320535,Electronic,La milonga.\nLadies and gentlemen\nA man.\nMil...,Other
4,248213,Pop,Anybody ever tell you that you're not whole\nA...,Pop
5,130989,Indie,I give myself up every day\nTo fight this woe ...,Pop
6,281194,Rock,We were in a park and I could jump so far\nSee...,Indie
7,312266,Rock,Guilty As A Saint\n(Prophet/klipschutz)\nI was...,Indie
8,103107,Folk,Once I had a darling mother though I can't rec...,Folk
9,116502,Rock,"I am a vision, I am justice\nNever thought tha...",Rock


#### Evaluating the few-shot prompting performance 

Below, we specify the true values and the predictions, and use them to make a classification report using sklearn. This gives us an overview of performence (precision, recall, f1) per genre as well as the total average weighted metrics.

In [109]:
true = fewshot['genre'] # retrieve true values
pred = fewshot['predicted_genre'] #retrieve predictions

# generate the classification report using sklearn
report = classification_report(
    true,
    pred,
    labels=genres,
    target_names=genres,
    output_dict=True
)

report = pd.DataFrame(report).transpose() # dict to df, transpose for visual satisfaction
report.to_csv("results/fewshot_report.csv", index=True) # save the report as csv
report

,precision,recall,f1-score,support
Hip-Hop,0.680851,0.785276,0.729345,163.0
Country,0.416667,0.393258,0.404624,89.0
Rock,0.740741,0.061069,0.112835,655.0
Electronic,0.307692,0.070175,0.114286,57.0
Pop,0.337255,0.337255,0.337255,255.0
Indie,0.017828,0.687500,0.034755,16.0
Folk,0.075000,0.352941,0.123711,17.0
R&B,0.151515,0.217391,0.178571,23.0
Metal,0.613445,0.483444,0.540741,151.0
Jazz,0.285714,0.046512,0.080000,43.0
